# Workout Time Series — TF LSTM 1RM forecast

Forecasts next-session weight for a (user, exercise) from the last 8
logged sessions. Real data comes from `export_ai_training_data` ->
`../data/user/workouts.csv`; when absent, a deterministic linear-progression
synthetic set is generated so the pipeline runs anywhere. Exported ONNX input is
`(None, 8, 1)` and outputs the predicted weight.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


In [ ]:
# Load real workouts or synthesize linear-progression demos
import os
import numpy as np
import pandas as pd
from pathlib import Path

wk_path = Path('../data/user/workouts.csv')
if wk_path.exists():
    w = pd.read_csv(wk_path)
    w['date'] = pd.to_datetime(w['date'])
    print('REAL workouts:', w.shape)
else:
    rng = np.random.default_rng(11)
    rows = []
    for u in range(40):
        for ex in ['Bench Press', 'Squat', 'Deadlift', 'OHP', 'Row']:
            base = 40 + rng.integers(0, 80)
            for wk in range(20):
                weight = base + 2.2 * wk + rng.normal(0, 2.5)
                rows.append({'user_id': u, 'exercise': ex, 'week': wk,
                             'weight_kg': round(float(weight), 1)})
    w = pd.DataFrame(rows)
    print('SYNTHETIC workouts:', w.shape,
          '(real data: python manage.py export_ai_training_data)')
print(w.groupby('exercise')['weight_kg'].mean().round(1).to_dict())

In [ ]:
# Build sliding-window sequences (window=8 -> next value)
import numpy as np
import tensorflow as tf

WINDOW = 8
seqs, targets = [], []
for (u, ex), g in w.groupby(['user_id', 'exercise']):
    g = g.sort_values('week')['weight_kg'].to_numpy(dtype=np.float32)
    if len(g) <= WINDOW:
        continue
    for i in range(len(g) - WINDOW):
        seqs.append(g[i:i + WINDOW])
        targets.append(g[i + WINDOW])
X = np.stack(seqs)[..., None]           # (N, 8, 1)
y = np.array(targets, dtype=np.float32)

split = int(0.8 * len(X))
Xtr, Xva, ytr, yva = X[:split], X[split:], y[:split], y[split:]
print('sequences:', X.shape, '| train:', len(ytr), '| val:', len(yva))

In [ ]:
# LSTM forecaster (8-step window -> 1 value)
inp = tf.keras.Input(shape=(WINDOW, 1))
x = tf.keras.layers.LSTM(32)(inp)
x = tf.keras.layers.Dense(16, activation='relu')(x)
out = tf.keras.layers.Dense(1)(x)
m = tf.keras.Model(inp, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'mse')
EPOCHS = {'smoke': 1, 'demo': 8, 'full': 25}[SCALE]
m.fit(Xtr, ytr, epochs=EPOCHS, batch_size=128,
      validation_data=(Xva, yva), verbose=1)

In [ ]:
# Evaluate: MAE / MAPE on held-out sequences
from sklearn.metrics import mean_absolute_error
pred = m.predict(Xva, batch_size=256)[:, 0]
mae = float(mean_absolute_error(yva, pred))
mape = float(np.mean(np.abs((yva - pred) / (yva + 1e-6))))
print(f'val MAE={mae:.2f} kg | MAPE={mape*100:.2f}%')
print('sample pred vs actual:')
pd.DataFrame({'actual': yva[:5].round(1), 'pred': pred[:5].round(1)}).to_string(index=False)

### Export contract (consumed by the AI service)

The cells below write `../models/workout_forecast.onnx` and its dynamic-INT8 quantized copy
`workout_forecast_int8.onnx`. `app/ml/serving.py::load_preferred('workout_forecast')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [ ]:
# Export ONNX (+ INT8)
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(m, Path('../models'), 'workout_forecast', '1.0.0')
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'workout_forecast', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'val_mae_kg': round(mae, 3), 'val_mape': round(mape, 4)}})
print('exported', q)